<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment41_Tamper_Evident_Custody_Chain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# EXPERIMENT 3
# A TAMPER-EVIDENT CHAIN OF CUSTODY REGISTER
# Hash-Linked Digital Evidence Ledger
# ============================================================

import hashlib
import json
from datetime import datetime, timezone, timedelta


# ============================================================
# 1. INDIAN STANDARD TIME (IST)
# ============================================================

IST = timezone(timedelta(hours=5, minutes=30))


# ============================================================
# 2. CREATE SHA-256 HASH FOR EACH ENTRY
# ============================================================

def _digest(entry):

    # Remove entry_hash before calculating the hash
    body = {
        k: v
        for k, v in entry.items()
        if k != "entry_hash"
    }

    canonical_data = json.dumps(
        body,
        sort_keys=True
    ).encode()

    return hashlib.sha256(
        canonical_data
    ).hexdigest()


# ============================================================
# 3. CUSTODY LEDGER CLASS
# ============================================================

class CustodyLedger:

    def __init__(self, case_id):

        self.case_id = case_id
        self.records = []


    # --------------------------------------------------------
    # ADD A CUSTODY EVENT
    # --------------------------------------------------------

    def add(
        self,
        exhibit_id,
        action,
        handled_by,
        location,
        purpose,
        ts=None
    ):

        # First record points to genesis hash
        if self.records:

            prev = self.records[-1]["entry_hash"]

        else:

            prev = "0" * 64


        entry = {

            "seq": len(self.records) + 1,

            "case_id": self.case_id,

            "exhibit_id": exhibit_id,

            "timestamp_ist": (
                ts or datetime.now(IST)
            ).isoformat(),

            "action": action,

            "handled_by": handled_by,

            "location": location,

            "purpose": purpose,

            "prev_hash": prev
        }


        # Generate current record hash
        entry["entry_hash"] = _digest(entry)


        self.records.append(entry)

        return entry


    # --------------------------------------------------------
    # VERIFY THE COMPLETE CHAIN
    # --------------------------------------------------------

    def verify(self):

        problems = []

        # Genesis hash
        prev = "0" * 64


        for r in self.records:

            # Check previous-record link
            if r["prev_hash"] != prev:

                problems.append(
                    f"seq {r['seq']}: broken link "
                    f"(prev_hash does not match "
                    f"previous record)"
                )


            # Check whether record content was modified
            if r["entry_hash"] != _digest(r):

                problems.append(
                    f"seq {r['seq']}: "
                    f"content modified after recording"
                )


            # Current hash becomes previous hash
            prev = r["entry_hash"]


        return (
            len(problems) == 0,
            problems
        )


    # --------------------------------------------------------
    # DISPLAY LEDGER AS TABLE
    # --------------------------------------------------------

    def to_table(self):

        header = (
            f"{'Seq':<5}"
            f"{'Exhibit':<18}"
            f"{'Timestamp (IST)':<28}"
            f"{'Action':<15}"
            f"{'Handled By':<22}"
            f"{'Location'}"
        )

        output = [header]

        output.append("-" * 120)


        for r in self.records:

            output.append(
                f"{r['seq']:<5}"
                f"{r['exhibit_id']:<18}"
                f"{r['timestamp_ist']:<28}"
                f"{r['action']:<15}"
                f"{r['handled_by']:<22}"
                f"{r['location']}"
            )


        return "\n".join(output)


# ============================================================
# 4. TEST CASES
# ============================================================

def run_tests():

    # --------------------------------------------------------
    # Create case
    # --------------------------------------------------------

    L = CustodyLedger(
        "CASE/CYB/2026/0417"
    )


    base = datetime(
        2026,
        8,
        20,
        9,
        0,
        tzinfo=IST
    )


    # --------------------------------------------------------
    # Add four custody events
    # --------------------------------------------------------

    L.add(
        "EX-01 Laptop",
        "SEIZED",
        "SI R. Menon",
        "Branch office, Chennai",
        "Seizure under panchnama",
        base
    )


    L.add(
        "EX-01 Laptop",
        "TRANSFERRED",
        "HC K. Das",
        "Evidence store, Chennai",
        "Secure storage",
        base + timedelta(hours=2)
    )


    L.add(
        "EX-01 Laptop",
        "IMAGED",
        "Examiner A. Rao",
        "Forensic lab, Bengaluru",
        "Bit-stream imaging with write blocker",
        base + timedelta(days=1)
    )


    L.add(
        "EX-01 Laptop",
        "ANALYSED",
        "Examiner A. Rao",
        "Forensic lab, Bengaluru",
        "Artefact analysis on working copy",
        base + timedelta(days=2)
    )


    # ========================================================
    # DISPLAY ORIGINAL LEDGER
    # ========================================================

    print("=" * 120)
    print("CHAIN OF CUSTODY REGISTER")
    print("=" * 120)

    print(L.to_table())

    print("\n")


    # ========================================================
    # TEST RESULTS
    # ========================================================

    results = []


    # --------------------------------------------------------
    # TC1 - Intact chain
    # --------------------------------------------------------

    ok, problems = L.verify()

    results.append(
        (
            "TC1 intact chain verifies",
            ok and problems == []
        )
    )


    # --------------------------------------------------------
    # TC2 - First record points to genesis
    # --------------------------------------------------------

    results.append(
        (
            "TC2 first record links to genesis",
            L.records[0]["prev_hash"]
            == "0" * 64
        )
    )


    # --------------------------------------------------------
    # TC3 - Every record links to previous record
    # --------------------------------------------------------

    links_correct = all(
        L.records[i]["prev_hash"]
        ==
        L.records[i - 1]["entry_hash"]
        for i in range(1, len(L.records))
    )


    results.append(
        (
            "TC3 record n links to record n-1",
            links_correct
        )
    )


    # --------------------------------------------------------
    # TC4 - Four records exist
    # --------------------------------------------------------

    results.append(
        (
            "TC4 four records present",
            len(L.records) == 4
        )
    )


    # --------------------------------------------------------
    # TC5 - Modify an existing record
    # --------------------------------------------------------

    L.records[1]["handled_by"] = "Unknown person"


    ok2, problems2 = L.verify()


    edit_detected = (
        ok2 is False
        and
        any(
            "modified" in p
            for p in problems2
        )
    )


    results.append(
        (
            "TC5 content edit detected",
            edit_detected
        )
    )


    # --------------------------------------------------------
    # TC6 - Delete a middle record
    # --------------------------------------------------------

    L2 = CustodyLedger(
        "CASE/CYB/2026/0418"
    )


    for i in range(4):

        L2.add(
            "EX-02 Phone",
            "ANALYSED",
            f"Examiner {i}",
            "Forensic Lab",
            "Analysis step",
            base + timedelta(hours=i)
        )


    # Delete record 3
    del L2.records[2]


    ok3, problems3 = L2.verify()


    deletion_detected = (
        ok3 is False
        and
        any(
            "broken link" in p
            for p in problems3
        )
    )


    results.append(
        (
            "TC6 deletion detected",
            deletion_detected
        )
    )


    # --------------------------------------------------------
    # TC7 - Empty ledger
    # --------------------------------------------------------

    empty_ledger = CustodyLedger("EMPTY-CASE")


    empty_valid = (
        empty_ledger.verify()[0]
        is True
    )


    results.append(
        (
            "TC7 empty ledger valid",
            empty_valid
        )
    )


    # ========================================================
    # PRINT TEST RESULTS
    # ========================================================

    print("=" * 70)
    print("CHAIN OF CUSTODY TEST RESULTS")
    print("=" * 70)


    for name, passed in results:

        print(
            f"{name:<40} -> "
            f"{'PASS' if passed else 'FAIL'}"
        )


    passed_count = sum(
        1
        for _, passed in results
        if passed
    )


    print("-" * 70)


    print(
        f"RESULT: {passed_count}/{len(results)} "
        "test cases passed"
    )


    print("=" * 70)


    return (
        passed_count == len(results)
    )


# ============================================================
# 5. RUN EXPERIMENT
# ============================================================

run_tests()


# ============================================================
# 6. SHOW HASH VALUES
# ============================================================

print("\n")
print("=" * 70)
print("HASH-LINK INFORMATION")
print("=" * 70)


# Create a fresh ledger for display

demo = CustodyLedger(
    "CASE/CYB/2026/0417"
)


demo.add(
    "EX-01 Laptop",
    "SEIZED",
    "SI R. Menon",
    "Branch Office, Chennai",
    "Initial seizure",
    datetime(2026, 8, 20, 9, 0, tzinfo=IST)
)


demo.add(
    "EX-01 Laptop",
    "TRANSFERRED",
    "HC K. Das",
    "Evidence Store, Chennai",
    "Secure storage",
    datetime(2026, 8, 20, 11, 0, tzinfo=IST)
)


demo.add(
    "EX-01 Laptop",
    "IMAGED",
    "Examiner A. Rao",
    "Forensic Lab, Bengaluru",
    "Forensic imaging",
    datetime(2026, 8, 21, 9, 0, tzinfo=IST)
)


demo.add(
    "EX-01 Laptop",
    "ANALYSED",
    "Examiner A. Rao",
    "Forensic Lab, Bengaluru",
    "Evidence analysis",
    datetime(2026, 8, 22, 9, 0, tzinfo=IST)
)


for record in demo.records:

    print("\nRecord:", record["seq"])

    print(
        "Previous Hash:",
        record["prev_hash"]
    )

    print(
        "Entry Hash:",
        record["entry_hash"]
    )


# ============================================================
# 7. FINAL VERIFICATION
# ============================================================

print("\n")
print("=" * 70)
print("FINAL CHAIN VERIFICATION")
print("=" * 70)


final_ok, final_problems = demo.verify()


if final_ok:

    print("CHAIN STATUS: INTACT")
    print("All custody records are cryptographically linked.")

else:

    print("CHAIN STATUS: BROKEN")

    for problem in final_problems:

        print(problem)


print("\n")
print("=" * 70)
print("EXPERIMENT 3 COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "A hash-linked chain of custody register was implemented."
)

print(
    "Record modification and deletion were detected."
)

print(
    "The integrity of the custody trail was verified."
)

print("=" * 70)

CHAIN OF CUSTODY REGISTER
Seq  Exhibit           Timestamp (IST)             Action         Handled By            Location
------------------------------------------------------------------------------------------------------------------------
1    EX-01 Laptop      2026-08-20T09:00:00+05:30   SEIZED         SI R. Menon           Branch office, Chennai
2    EX-01 Laptop      2026-08-20T11:00:00+05:30   TRANSFERRED    HC K. Das             Evidence store, Chennai
3    EX-01 Laptop      2026-08-21T09:00:00+05:30   IMAGED         Examiner A. Rao       Forensic lab, Bengaluru
4    EX-01 Laptop      2026-08-22T09:00:00+05:30   ANALYSED       Examiner A. Rao       Forensic lab, Bengaluru


CHAIN OF CUSTODY TEST RESULTS
TC1 intact chain verifies                -> PASS
TC2 first record links to genesis        -> PASS
TC3 record n links to record n-1         -> PASS
TC4 four records present                 -> PASS
TC5 content edit detected                -> PASS
TC6 deletion detected           